# Исследование надежности заемщиков


Во второй части проекта вы выполните шаги 3 и 4. Их вручную проверит ревьюер.
Чтобы вам не пришлось писать код заново для шагов 1 и 2, мы добавили авторские решения в ячейки с кодом. 



## Откройте таблицу и изучите общую информацию о данных

**Задание 1. Импортируйте библиотеку pandas. Считайте данные из csv-файла в датафрейм и сохраните в переменную `data`. Путь к файлу:**

`/datasets/data.csv`

In [1]:
import pandas as pd

try:
    data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

**Задание 2. Выведите первые 20 строчек датафрейма `data` на экран.**

In [2]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


**Задание 3. Выведите основную информацию о датафрейме с помощью метода `info()`.**

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Предобработка данных

### Удаление пропусков

**Задание 4. Выведите количество пропущенных значений для каждого столбца. Используйте комбинацию двух методов.**

In [4]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

**Задание 5. В двух столбцах есть пропущенные значения. Один из них — `days_employed`. Пропуски в этом столбце вы обработаете на следующем этапе. Другой столбец с пропущенными значениями — `total_income` — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца `income_type`. Например, у человека с типом занятости `сотрудник` пропуск в столбце `total_income` должен быть заполнен медианным доходом среди всех записей с тем же типом.**

In [5]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

### Обработка аномальных значений

**Задание 6. В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. таким артефактом будет отрицательное количество дней трудового стажа в столбце `days_employed`. Для реальных данных это нормально. Обработайте значения в этом столбце: замените все отрицательные значения положительными с помощью метода `abs()`.**

In [6]:
data['days_employed'] = data['days_employed'].abs()

**Задание 7. Для каждого типа занятости выведите медианное значение трудового стажа `days_employed` в днях.**

In [7]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставьте их как есть. Тем более этот столбец не понадобится вам для исследования.

**Задание 8. Выведите перечень уникальных значений столбца `children`.**

In [8]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

**Задание 9. В столбце `children` есть два аномальных значения. Удалите строки, в которых встречаются такие аномальные значения из датафрейма `data`.**

In [9]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

**Задание 10. Ещё раз выведите перечень уникальных значений столбца `children`, чтобы убедиться, что артефакты удалены.**

In [10]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

### Удаление пропусков (продолжение)

**Задание 11. Заполните пропуски в столбце `days_employed` медианными значениями по каждого типа занятости `income_type`.**

In [11]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

**Задание 12. Убедитесь, что все пропуски заполнены. Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.**

In [12]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Изменение типов данных

**Задание 13. Замените вещественный тип данных в столбце `total_income` на целочисленный с помощью метода `astype()`.**

In [13]:
data['total_income'] = data['total_income'].astype(int)

### Обработка дубликатов

**Задание 14. Обработайте неявные дубликаты в столбце `education`. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведите их к нижнему регистру. Проверьте остальные столбцы.**

In [14]:
data['education'] = data['education'].str.lower()

**Задание 15. Выведите на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалите их.**

In [15]:
data.duplicated().sum()

71

In [16]:
data = data.drop_duplicates()

### Категоризация данных

**Задание 16. На основании диапазонов, указанных ниже, создайте в датафрейме `data` столбец `total_income_category` с категориями:**

- 0–30000 — `'E'`;
- 30001–50000 — `'D'`;
- 50001–200000 — `'C'`;
- 200001–1000000 — `'B'`;
- 1000001 и выше — `'A'`.


**Например, кредитополучателю с доходом 25000 нужно назначить категорию `'E'`, а клиенту, получающему 235000, — `'B'`. Используйте собственную функцию с именем `categorize_income()` и метод `apply()`.**

In [17]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [18]:
data['total_income_category'] = data['total_income'].apply(categorize_income)

**Задание 17. Выведите на экран перечень уникальных целей взятия кредита из столбца `purpose`.**

In [19]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

**Задание 18. Создайте функцию, которая на основании данных из столбца `purpose` сформирует новый столбец `purpose_category`, в который войдут следующие категории:**

- `'операции с автомобилем'`,
- `'операции с недвижимостью'`,
- `'проведение свадьбы'`,
- `'получение образования'`.

**Например, если в столбце `purpose` находится подстрока `'на покупку автомобиля'`, то в столбце `purpose_category` должна появиться строка `'операции с автомобилем'`.**

**Используйте собственную функцию с именем `categorize_purpose()` и метод `apply()`. Изучите данные в столбце `purpose` и определите, какие подстроки помогут вам правильно определить категорию.**

In [20]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [21]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

### Шаг 3. Исследуйте данные и ответьте на вопросы

#### 3.1 Есть ли зависимость между количеством детей и возвратом кредита в срок?

In [22]:
# построим сводную таблицу с группировкой количества детей по возврату кредита в срок
# сразу посчитаем количество общее для каждой группы, и количество тех, кто задерживал возврат
ratio_of_children = data.pivot_table(index = 'children', values = 'debt', aggfunc = {'sum','count'})

# переименуем колонки 
ratio_of_children = ratio_of_children.rename(columns ={'count':'total','sum':'target'})

# посчитаем % невозврата кредита в срок
ratio_of_children['ratio'] = round(ratio_of_children['target']/ratio_of_children['total']*100,2)

# отсортируем данные и обновим индексы
ratio_of_children = ratio_of_children.sort_values('ratio').reset_index()

ratio_of_children

,children,total,target,ratio
0,5,9,0,0.00
1,0,14091,1063,7.54
2,3,330,27,8.18
3,1,4808,444,9.23
4,2,2052,194,9.45
5,4,41,4,9.76


Сразу видно, что в строке 0 недостаточно данных для поиска зависимости. Большая часть данных сгруппирована в строках 1,3,4. 

In [23]:
# посчитаем часть выборки для выявления корреляции
sample_size = ratio_of_children.loc[[1,3,4],'total'].sum()/ratio_of_children['total'].sum()
# посчитаем корреляцию
corr_sample_size = ratio_of_children.loc[[1,3,4],'ratio'].corr(ratio_of_children.loc[[1,3,4],'children'])

print(f'''Строки 1,3,4 содержат {sample_size:.3%} выборки.
Корреляция для этих данных составляет {corr_sample_size:.3}''')
ratio_of_children.loc[[1,3,4]]

Строки 1,3,4 содержат 98.219% выборки.
Корреляция для этих данных составляет 0.914


,children,total,target,ratio
1,0,14091,1063,7.54
3,1,4808,444,9.23
4,2,2052,194,9.45


**Вывод:**

К сожалению, данных о семьях с 3-мя, 4-мя, 5-ю детьми недостаточно для корректного анализа.

А вот для семей с 0,1,2 детьми, что составляют большую часть выборки - прослеживается зависимость: чем больше детей, тем больше % невозврата кредита в срок. Это можно обьяснить, например более частыми незапланированными тратами на детей, что в следствии откладывает платеж по кредиту и более стабильное финансовое состояние у тех, у кого нет детей. 

Корреляция показывает, что присутствует положительная связь: с повышением количества детей - увеличивается % невозврата кредита в срок.

В связи с тем, что в этих категориях семей содержится  больше 98% выборки можно предположить, что данная зависимость распростроняется на все категории.

#### 3.2 Есть ли зависимость между семейным положением и возвратом кредита в срок?

In [24]:
# построим сводную таблицу с группировкой семейного положения по возврату кредита в срок
# сразу посчитаем количество общее для каждой группы, и количество тех, кто задерживал возврат
ratio_of_family_status = data.pivot_table(index = 'family_status', values = 'debt', aggfunc = {'sum','count'})

# переименуем колонки 
ratio_of_family_status = ratio_of_family_status.rename(columns ={'count':'total','sum':'target'})

# посчитаем % невозврата кредита в срок
ratio_of_family_status['ratio'] = round(ratio_of_family_status['target']/ratio_of_family_status['total']*100,2)

# отсортируем данные
ratio_of_family_status = ratio_of_family_status.sort_values('ratio')

ratio_of_family_status

,total,target,ratio
family_status,,,
вдовец / вдова,951,63,6.62
в разводе,1189,84,7.06
женат / замужем,12261,927,7.56
гражданский брак,4134,385,9.31
Не женат / не замужем,2796,273,9.76


**Вывод:**

Данных достаточно, для того, чтобы сделать вывод, что те, кто уже побывал в браке или состоит в нем, задерживают выплаты реже, чем те, кто не женаты / не замужем или в гражданском браке

#### 3.3 Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

In [25]:
# построим сводную таблицу с группировкой уровня дохода по возврату кредита в срок
# сразу посчитаем количество общее для каждой группы, и количество тех, кто задерживал возврат
ratio_of_income_category = data.pivot_table(index = 'total_income_category', values = 'debt', aggfunc = {'sum','count'})

# переименуем колонки 
ratio_of_income_category = ratio_of_income_category.rename(columns ={'count':'total','sum':'target'})

# посчитаем % невозврата кредита в срок
ratio_of_income_category['ratio'] = round(ratio_of_income_category['target']/ratio_of_income_category['total']*100,2)

# отсортируем данные
ratio_of_income_category = ratio_of_income_category.sort_values('ratio')

ratio_of_income_category

,total,target,ratio
total_income_category,,,
D,349,21,6.02
B,5014,354,7.06
A,25,2,8.00
C,15921,1353,8.50
E,22,2,9.09


Строки 0, 2, 4 не содержат достаточного количество данных для определения зависимости по ним.

In [26]:
# посчитаем часть выборки для выявления зависимости
sample_size = ratio_of_income_category.loc[['B','C'],'total'].sum()/ratio_of_income_category['total'].sum()

print(f'''Строки B, C содержат {sample_size:.3%} выборки''')
ratio_of_income_category.loc[['B','C']]

Строки B, C содержат 98.144% выборки


,total,target,ratio
total_income_category,,,
B,5014,354,7.06
C,15921,1353,8.50


**Вывод:**

На основании того, что в категориях дохода семей B и C содержится более 98% всей выборки. Соответсвенно зависимость можно распространить и на остальные группы. Зависимость обратная: с увеличением дохода - снижается % невыплаты в срок. 

#### 3.4 Как разные цели кредита влияют на его возврат в срок?

In [27]:
# построим сводную таблицу с группировкой  категорий целей кредита по возврату кредита в срок
# сразу посчитаем количество общее для каждой группы, и количество тех, кто задерживал возврат
ratio_of_purpose_category = data.pivot_table(index = 'purpose_category', values = 'debt', aggfunc = {'sum','count'})

# переименуем колонки 
ratio_of_purpose_category = ratio_of_purpose_category.rename(columns ={'count':'total','sum':'target'})

# посчитаем % невозврата кредита в срок
ratio_of_purpose_category['ratio'] = round(ratio_of_purpose_category['target']/ratio_of_purpose_category['total']*100,2)

# отсортируем данные и обновим индексы
ratio_of_purpose_category = ratio_of_purpose_category.sort_values('ratio')

ratio_of_purpose_category

,total,target,ratio
purpose_category,,,
операции с недвижимостью,10751,780,7.26
проведение свадьбы,2313,183,7.91
получение образования,3988,369,9.25
операции с автомобилем,4279,400,9.35


**Вывод:** 

В случае с людьми, которые стремятся к созданию семьи или приобретению своей жил площадью, вероятность невозврата кредита заметно ниже, чем у людей, которые берут потребительский кредит на вещи существенно ниже по стоимости. Здесь, вероятно, играет роль величина суммы кредита

#### 3.5 Приведите возможные причины появления пропусков в исходных данных.

*Ответ:* 

Пропуски  были в столбцах `days_employed` и `total_income`. Человеческий фактор, кто-то не захотел отвечать, вероятно посчитав не нужным распространяться персональными данными. Количество пропусков в столбцах одинаковы, можно сделать вывод, что заполнение одного из столбцов влияет на заполненность другого столбца.

#### 3.6 Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

*Ответ:*

Медианное значение лучше всего отражает выборку в целом, в случае, если есть незаполненные данные. Среднее арифметическое отражает ситуацию некорректно, так как не учитывает разброс данных.

### Шаг 4: общий вывод.

В целом, по выдвинутым гипотезам корреляция прослеживается. По про проведенному анализу можно сделать вывод о том, что люди, которые с наибольшей вероятностью допустят просрочку по кредиту, вероятно, будут иметь 2 детей или более, возможно будут состоять в гражданском браке, а более вероятно, что не будут состоять в браке вовсе, и их доход будет существенно низок. Также, возможно, что целью кредита будет получение образования или покупка автомобиля, а также прочие относительно невысокие по стоимости товары и услуги. Не обязательно, что все эти факторы должны совпасть, но комбинация нескольких из них может говорить о том, что в будущем возможны просрочки по кредиту. Напротив, высока вероятность возврата кредита в срок, если человек более не состоит в браке, имеет высокий доход и, вероятно, у него нет детей на попечении.

Рекомендовал бы проанализировать зависимость между `типом занятости`; `уровнем образования` и `возвратом кредита в срок`. А так же рассмотреть комбинации факторов и их влияние на невозврат кредита в срок. Это даст более конкретное и детальное понимание о надежности заемщиков.